# Lab 01 — Interaction Linear Regression (THÂN)

**Nhiệm vụ:** Dự đoán `Post_Semester_GPA` bằng OLS và kiểm tra liệu liên hệ dự đoán giữa số giờ dùng GenAI và GPA có thay đổi theo `Primary_Use_Case`, `Prompt_Engineering_Skill`, năng lực đầu kỳ hay thời gian học truyền thống.

Notebook này **tự chứa phần triển khai**, không cần import package `lab01_ml`, không dùng API và **không chia lại dữ liệu**. Ba file CSV đã có sẵn ở `data/train/`, `data/validation/`, `data/test/`.

> Dữ liệu có tính tổng hợp (synthetic). Hệ số hồi quy, xu hướng, kịch bản *what-if* chỉ thể hiện **liên hệ trong mô hình**, **không chứng minh quan hệ nhân quả**. Thời điểm đo các biến tâm lý chưa chắc phù hợp để dự báo trước học kỳ.

**Cách học:** đọc ô Markdown trước, chạy ô code ngay sau; không cần biết sâu về scikit-learn ngoài khái niệm Linear Regression. Các bảng/biểu đồ chỉ được tạo từ dữ liệu CSV thực sự tải lên trong lần chạy.

## 0. Hướng dẫn chạy và khóa tập test

1. Chọn kernel Python đã cài dependencies (`numpy`, `pandas`, `scikit-learn`, `matplotlib`, `seaborn`, `joblib`, `python-dotenv`; `ipywidgets` tùy chọn).
2. Đặt `LAB01_DATA_ROOT` trong `.env` của dự án hoặc biến môi trường, trỏ tới thư mục chứa ba CSV frozen.
3. **Restart Kernel → Run All** khi đã chốt quy trình. Mặc định notebook chạy **final test trên toàn bộ 7.500 mẫu** sau khi lựa chọn mô hình bằng CV và fit lại trên train+validation.
4. Nếu còn phát triển, đặt `LAB01_RUN_FINAL_TEST=0` trước khi khởi động kernel. Chế độ này chỉ báo cáo CV/validation.
5. `LAB01_SMOKE_TEST=1` chỉ dùng để kiểm tra luồng code với fixture, không cho phép final test; đặt `LAB01_RUN_FINAL_TEST=0` nếu đang bật nó rõ ràng.
6. Registry trong `outputs/interaction` ngăn chạy lại final test trên cùng bộ dữ liệu. Nếu đã đánh giá, dùng kết quả từ RUN_ID đã lưu; không xóa registry để tuning. Nếu cần thay đổi mô hình sau khi xem test, cần một tập test độc lập mới.

```text
Lab01/
├── .env
├── data/
│   ├── train/dataset.csv
│   ├── validation/dataset.csv
│   └── test/dataset.csv
├── notebooks/interaction_linear_regression/interaction_linear_regression.ipynb
└── outputs/interaction/runs/<run-id>/  # tự sinh
```

In [ ]:
from pathlib import Path
import os, sys, hashlib, json, uuid, warnings
from datetime import datetime, timezone
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / ".env").is_file()), cwd)
load_dotenv(PROJECT_ROOT / ".env")
print('PROJECT_ROOT:', PROJECT_ROOT)
configured_root = os.getenv("LAB01_DATA_ROOT")
if not configured_root:
    raise ValueError("Chưa cấu hình LAB01_DATA_ROOT trong .env hoặc biến môi trường")
DATA_ROOT = Path(configured_root).expanduser()
if not DATA_ROOT.is_absolute():
    DATA_ROOT = PROJECT_ROOT / DATA_ROOT
DATA_ROOT = DATA_ROOT.resolve()
TRAIN_CSV = DATA_ROOT / "train" / "dataset.csv"
VAL_CSV = DATA_ROOT / "validation" / "dataset.csv"
TEST_CSV = DATA_ROOT / "test" / "dataset.csv"

# Kiểm tra đường dẫn
for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy: {path}")
# Khi chỉ cần thử code bằng dữ liệu kiểm thử nhân tạo: LAB01_SMOKE_TEST=1.
# Mặc định luôn đòi đúng 50.000 dòng và tỷ lệ 70/15/15.
SMOKE_TEST = os.environ.get('LAB01_SMOKE_TEST', '0') == '1'
RUN_FINAL_TEST = os.environ.get('LAB01_RUN_FINAL_TEST', '0' if SMOKE_TEST else '1') == '1'
if SMOKE_TEST and RUN_FINAL_TEST:
    raise ValueError('Smoke test không được chạy final test. Đặt LAB01_RUN_FINAL_TEST=0.')

# Output tách theo run-id; không tái sử dụng biểu đồ cũ hay output từ kernel cũ.
ROOT = DATA_ROOT.parent if DATA_ROOT.is_dir() else Path.cwd().resolve()
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:7]
OUT_ROOT = ROOT / 'outputs' / ('interaction_smoke' if SMOKE_TEST else 'interaction')
OUT = OUT_ROOT / 'runs' / RUN_ID
FIG, TABLES, MODELS = OUT / 'figures', OUT / 'tables', OUT / 'models'
for path in [FIG, TABLES, MODELS]: path.mkdir(parents=True, exist_ok=False if path==OUT else True)

# Imports chính, không yêu cầu dự án phải có src/ hoặc file .py riêng.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
SEED = 42
np.random.seed(SEED)
FIGURES_CREATED = []

def show(fig, name, observation):
    """Lưu duy nhất 1 file/figure và hiển thị trong notebook."""
    dest = FIG / (name + '.png')
    if dest.exists():
        raise RuntimeError(f'Figure name bị lặp: {dest}')
    fig.tight_layout()
    fig.savefig(dest, dpi=170, bbox_inches='tight')
    display(fig)
    plt.close(fig)
    FIGURES_CREATED.append(str(dest))
    print('Nhận xét:', observation)

print('DATA_ROOT:', DATA_ROOT)
print('RUN_ID:', RUN_ID)
print('RUN_FINAL_TEST:', RUN_FINAL_TEST, '| SMOKE_TEST:', SMOKE_TEST)
print('Output:', OUT)

## 1. Hợp đồng dữ liệu: X và y

**Target y (hồi quy):** `Post_Semester_GPA`, là GPA sau học kỳ (giá trị liên tục).

**12 input (X)** gồm:

| Nhóm | Columns | Xử lý |
|---|---|---|
| A — học tập | `Pre_Semester_GPA`, `Traditional_Study_Hours`, `Year_of_Study`, `Major_Category` | 2 numeric + 2 one-hot |
| B — GenAI | `Weekly_GenAI_Hours`, `Primary_Use_Case`, `Prompt_Engineering_Skill`, `Tool_Diversity`, `Paid_Subscription` | 3 numeric/binary + 2 one-hot |
| C — bối cảnh | `Institutional_Policy` | one-hot |
| D — tâm lý | `Perceived_AI_Dependency`, `Anxiety_Level_During_Exams` | one-hot; được ablate |

**Không đưa vào X:** `Student_ID` (định danh), `Post_Semester_GPA` (target), `Skill_Retention_Score`, `Burnout_Risk_Level`, `GPA_Change` nếu tồn tại. Riêng GPA Change = Post − Pre; sử dụng để dự đoán Post là rò rỉ nhãn.

Mỗi dòng input trả về **một số GPA dự đoán**. Các cột interaction được tự động tạo trong pipeline và **không yêu cầu người dùng nhập**.

In [ ]:
TARGET = 'Post_Semester_GPA'
H, P, S = 'Weekly_GenAI_Hours', 'Pre_Semester_GPA', 'Traditional_Study_Hours'
NUMERIC = [H, P, S, 'Tool_Diversity', 'Paid_Subscription']
CATEGORICAL = [
    'Year_of_Study', 'Major_Category', 'Primary_Use_Case',
    'Prompt_Engineering_Skill', 'Institutional_Policy',
    'Perceived_AI_Dependency', 'Anxiety_Level_During_Exams'
]
INPUTS = NUMERIC + CATEGORICAL
EXCLUDED = {'Student_ID', TARGET, 'Skill_Retention_Score',
            'Burnout_Risk_Level', 'GPA_Change'}
EXPECTED_TERMS = {0:39, 1:41, 2:45, 3:53, 4:55}
assert len(INPUTS) == 12 and not set(INPUTS) & EXCLUDED
print('12 raw inputs:', INPUTS)
print('Excluded from X:', sorted(EXCLUDED))

## 2. Đọc ba tập frozen, kiểm tra chống leakage

**Train 70% = 35.000 dòng, validation 15% = 7.500, test 15% = 7.500** (tổng 50.000 dòng, 16 cột gốc).

- Không dùng `train_test_split`, `StratifiedShuffleSplit` hay phép chia dữ liệu mới.
- Kiểm tra schema, missing values, `Student_ID` trùng giữa các tập, kiểm tra số dòng; lưu SHA-256 của **ba CSV** để cả nhóm dùng cùng phiên bản.
- Test chỉ được đọc để kiểm tra cấu trúc/ID; **chưa tính metric hay dùng y_test** cho đến khi chốt mô hình.
- Nếu các CSV có dữ liệu không hợp lệ, dừng với lỗi rõ ràng, **không giả tạo/điền dữ liệu**.

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def load_frozen_splits():
    paths = {'train': TRAIN_CSV, 'validation': VAL_CSV, 'test': TEST_CSV}
    absent = [str(p) for p in paths.values() if not p.is_file()]
    if absent:
        raise FileNotFoundError('Không tìm thấy CSV frozen. Kiểm tra DATA_ROOT:\n' + '\n'.join(absent))
    frames, info = {}, {'run_id': RUN_ID, 'type': 'pre_existing_frozen_csv',
                         'smoke_test': SMOKE_TEST, 'sha256': {}, 'counts': {}}
    for split, path in paths.items():
        frame = pd.read_csv(path)
        required = set(INPUTS) | {TARGET, 'Student_ID'}
        missing = sorted(required - set(frame.columns))
        if missing: raise ValueError(f'{split}: thiếu cột: {missing}')
        if frame.columns.duplicated().any(): raise ValueError(f'{split}: tên cột bị trùng')
        if frame[INPUTS+[TARGET,'Student_ID']].isna().any().any():
            raise ValueError(f'{split}: có missing value trong input, ID hoặc target; kiểm tra trước khi train')
        frame = frame.reset_index(drop=True)
        frames[split] = frame
        info['counts'][split] = len(frame)
        info['sha256'][split] = sha256_file(path)
        print(split, 'shape=', frame.shape, 'SHA256=', info['sha256'][split][:14]+'…')
    all_ids = pd.concat([f['Student_ID'].astype(str) for f in frames.values()], ignore_index=True)
    if all_ids.duplicated().any():
        examples = all_ids[all_ids.duplicated(keep=False)].head().tolist()
        raise ValueError(f'Student_ID trùng trong/giữa các split: {examples}')
    if not SMOKE_TEST:
        exact = {'train':35000,'validation':7500,'test':7500}
        if info['counts'] != exact:
            raise ValueError(f'Frozen split không đúng 70/15/15 của 50k: {info["counts"]} != {exact}')
        widths = {sp:f.shape[1] for sp,f in frames.items()}
        if any(w != 16 for w in widths.values()):
            raise ValueError(f'Các file phải có 16 cột dữ liệu gốc, thực tế: {widths}')
    if any(set(f.columns) != set(frames['train'].columns) for f in frames.values()):
        raise ValueError('Ba split có schema cột khác nhau.')
    (OUT / 'split_provenance.json').write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding='utf-8')
    return frames, info

frames, provenance = load_frozen_splits()
train, val = frames['train'], frames['validation']
print('Split frozen đúng:', provenance['counts'])
if SMOKE_TEST:
    print('⚠️ SMOKE TEST: mọi con số chỉ kiểm tra luồng code, KHÔNG phải kết quả Lab 01.')

## 3. EDA chỉ trên tập train

Đồ thị khám phá dữ liệu được tạo từ **train** để giữ nguyên tính độc lập của validation và test. Quan sát giá trị trung bình theo nhóm chỉ là thống kê chưa điều chỉnh, **không phải ước lượng tương tác đã kiểm soát biến khác**.

In [ ]:
print('Dtype & missing của train:')
display(pd.DataFrame({'dtype':train.dtypes.astype(str), 'missing':train.isna().sum(),
                      'n_unique':train.nunique()}).sort_index())
print('Tóm tắt target & input numeric:')
display(train[[P,H,S,'Tool_Diversity',TARGET]].describe().T.round(3))

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
ax.hist(train[TARGET],bins=35,edgecolor='white')
ax.set(title='01 · Phân bố GPA sau học kỳ (train)',xlabel='Post Semester GPA',ylabel='Số sinh viên')
show(fig,'01_train_gpa',f"Trung vị={train[TARGET].median():.3f}; tỷ lệ GPA=4.0 là {(train[TARGET]>=4).mean():.2%}.")

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
ax.hist(train[H],bins=35,edgecolor='white')
ax.set(title='02 · Phân bố số giờ dùng GenAI (train)',xlabel='Giờ/tuần',ylabel='Số sinh viên')
show(fig,'02_train_ai_hours',f"Median={train[H].median():.2f}; Q1={train[H].quantile(.25):.2f}; Q3={train[H].quantile(.75):.2f}.")

**EDA kết hợp:** `GPA Change = Post GPA − Pre GPA` chỉ phục vụ mô tả trên train, tuyệt đối không là predictor. Biểu đồ dưới lấy mẫu điểm để tránh đè hàng chục nghìn marker.

In [ ]:
changes = train[TARGET] - train[P]
ix = train.sample(n=min(len(train),2500),random_state=SEED).index
fig,ax=plt.subplots(figsize=(8,4))
ax.scatter(train.loc[ix,H],changes.loc[ix],s=9,alpha=.22)
ax.set(title='03 · GenAI Hours vs GPA Change (train only)',xlabel='GenAI Hours',ylabel='GPA Change (EDA-only)')
show(fig,'03_train_gpa_change',f"Tương quan Pearson(train)={np.corrcoef(train[H], changes)[0,1]:.3f}, không suy ra nhân quả.")

In [ ]:
means = train.pivot_table(index='Primary_Use_Case',columns='Prompt_Engineering_Skill',
                          values=TARGET,aggfunc='mean')
fig,ax=plt.subplots(figsize=(9,4))
sns.heatmap(means,annot=True,fmt='.3f',cmap='Blues',ax=ax)
ax.set(title='04 · GPA trung bình theo Use Case × Prompt Skill (train)',xlabel='Prompt Skill',ylabel='Use Case')
show(fig,'04_group_means',f"Khoảng trung bình nhóm: {np.nanmin(means.values):.3f}–{np.nanmax(means.values):.3f}. Đây là thống kê thô.")

## 4. Thuật toán: OLS và interaction khác gì Linear Regression thường?

OLS tối ưu **tổng bình phương phần dư**:

\[
\hat\beta=\arg\min_{\beta}\sum_{i=1}^n (y_i-\hat y_i)^2,
\qquad \hat y_i=\beta_0+\sum_j\beta_j x_{ij}.
\]

Không có tương tác: ảnh hưởng dự đoán của thêm 1 giờ AI là **cùng một hệ số** đối với các Use Case/Skill khác nhau.

Có tương tác: thêm tích giữa các feature. **Vẫn là Linear Regression** vì tuyến tính theo tham số \(\beta\), dù \(H_c K\) là phép nhân hai biến đầu vào.

Ví dụ với \(K=1\) nếu Beginner, \(K=0\) nếu Advanced:

\[
\hat y=2.5+0.01H_c+0.1K-0.006(H_cK).
\]

- Advanced: \(\partial\hat y/\partial H = 0.01\) GPA/giờ.
- Beginner: \(0.01-0.006=0.004\) GPA/giờ.
- Con số trên **chỉ minh họa phép tính**, không được dùng làm kết quả từ dataset.

**Hierarchy:** nếu có tích \(H_cK\), phải giữ cả \(H_c\) và \(K\) trong mô hình. Không tạo dummy tự do rồi nhân mọi cột một cách mù quáng.

**Centering:** \(H_c=H-\overline H_{train}\), \(P_c=P-\overline P_{train}\), \(S_c=S-\overline S_{train}\). Mean học từ tập huấn luyện (hoặc 4/5 training portion trong mỗi CV fold). Việc này giúp hệ số chính diễn giải tại mức trung bình của các biến liên tục.

**One-Hot với reference category:** 3 mức Prompt Skill tạo 2 dummy; 5 Use Case tạo 4 dummy; nhóm tham chiếu là mức bị bỏ trong `drop='first'`. Với intercept, không giữ cả k dummy để tránh đa cộng tuyến hoàn hảo.

## 5. Năm cấu hình I0–I4

Ký hiệu \(H_c,P_c,S_c\) là các biến center; \(U_1\ldots U_4\) là dummy Use Case; \(K_1,K_2\) là dummy Prompt Skill.

\[
\begin{aligned}
I0: &\quad \hat y=\beta_0+\beta^T X_{main} \\
I1: &\quad I0+\sum_{j=1}^{2}\gamma_j H_cK_j \\
I2: &\quad I1+\sum_{i=1}^{4}\delta_i H_cU_i \\
I3: &\quad I2+\sum_{i=1}^{4}\sum_{j=1}^{2}\eta_{ij}U_iK_j \\
I4: &\quad I3+\theta H_cP_c+\phi H_cS_c.
\end{aligned}
\]

| Cấu hình | Tương tác bổ sung | Predictor terms (không tính intercept) |
|---|---|---:|
| I0 | Main effects | 39 |
| I1 | AI Hours × Prompt Skill (+2) | 41 |
| I2 | AI Hours × Use Case (+4) | 45 |
| **I3** | Use Case × Prompt Skill (+8) | **53** |
| I4 | AI Hours × Pre GPA và Study Hours (+2) | 55 |

**I3 là cấu hình nghiên cứu chính**, I4 dùng so sánh mở rộng. I0–I4 dùng **cùng OLS**, chỉ khác cách xây dựng X. **Không thêm H² hoặc spline**, vì chúng là hướng nghiên cứu của các thành viên khác.

In [ ]:
class InteractionFeatures(BaseEstimator, TransformerMixin):
    """Biến 12 raw inputs thành ma trận thiết kế reference-coded + interactions.

    `model=0..4`. `drop_columns` dành riêng cho ablation, không làm đổi raw input ở UI.
    Không sử dụng thống kê từ validation/test khi fit transformer.
    """
    def __init__(self, model=3, drop_columns=()):
        self.model = model
        self.drop_columns = drop_columns

    def _normalize(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError('Đầu vào phải là pandas DataFrame với tên cột raw.')
        missing = set(self.active_inputs_) - set(X.columns)
        if missing: raise ValueError('Thiếu raw input features: ' + str(sorted(missing)))
        frame = X.loc[:,self.active_inputs_].copy()
        for col in self.numeric_columns_:
            if col == 'Paid_Subscription':
                st = frame[col].astype(str).str.strip().str.lower()
                mapping = {'true':1.,'false':0.,'1':1.,'0':0.,'yes':1.,'no':0.}
                frame[col] = st.map(mapping)
                if frame[col].isna().any():
                    raise ValueError('Paid_Subscription phải là bool hoặc 0/1.')
            else:
                frame[col] = pd.to_numeric(frame[col], errors='raise')
            if not np.isfinite(frame[col].astype(float).to_numpy()).all():
                raise ValueError(f'{col}: NaN/Inf trong dữ liệu số.')
        for col in self.cat_columns_:
            if frame[col].isna().any(): raise ValueError(f'{col}: thiếu giá trị category')
            # Làm nhất quán categorical ordinal 1, 2, ... dù CSV parse là số.
            if col in {'Perceived_AI_Dependency','Anxiety_Level_During_Exams'}:
                frame[col] = frame[col].astype(int).astype(str)
            else:
                frame[col] = frame[col].astype(str)
        return frame

    def fit(self, X, y=None):
        if self.model not in range(5): raise ValueError('model phải là 0,1,2,3,4')
        drop = set(self.drop_columns)
        if drop - set(INPUTS): raise ValueError(f'drop_columns không hợp lệ: {drop-set(INPUTS)}')
        # Giữ nguyên signature 12 raw fields, trừ các term ablation.
        self.active_inputs_ = [col for col in INPUTS if col not in drop]
        self.numeric_columns_ = [col for col in NUMERIC if col not in drop]
        self.cat_columns_ = [col for col in CATEGORICAL if col not in drop]
        if any(col not in self.active_inputs_ for col in [H,P,S]):
            raise ValueError('Các tương tác I0–I4 cần H, P, S; không drop chúng ở ablation.')
        df = self._normalize(X)
        self.means_ = {c:float(df[c].mean()) for c in [H,P,S]}
        try:
            self.encoder_ = OneHotEncoder(drop='first',handle_unknown='error',sparse_output=False)
        except TypeError: # sklearn <1.2
            self.encoder_ = OneHotEncoder(drop='first',handle_unknown='error',sparse=False)
        self.encoder_.fit(df[self.cat_columns_])
        self.references_ = {c:str(cats[0]) for c,cats in zip(self.cat_columns_,self.encoder_.categories_)}
        self.dummy_names_ = list(self.encoder_.get_feature_names_out(self.cat_columns_))
        self.base_names_ = [f'{c}_centered' if c in [H,P,S] else c for c in self.numeric_columns_] + self.dummy_names_
        self.feature_names_ = list(self.base_names_)
        u_names = [col for col in self.dummy_names_ if col.startswith('Primary_Use_Case_')]
        k_names = [col for col in self.dummy_names_ if col.startswith('Prompt_Engineering_Skill_')]
        self.u_names_,self.k_names_ = u_names,k_names
        if self.model >= 1:
            self.feature_names_ += [f'Hc*{k}' for k in k_names]
        if self.model >= 2:
            self.feature_names_ += [f'Hc*{u}' for u in u_names]
        if self.model >= 3:
            self.feature_names_ += [f'{u}*{k}' for u in u_names for k in k_names]
        if self.model >= 4:
            self.feature_names_ += ['Hc*Pc','Hc*Sc']
        self.n_features_out_ = len(self.feature_names_)
        return self

    def transform(self, X):
        df = self._normalize(X)
        a = df[self.numeric_columns_].to_numpy(dtype=float).copy()
        for col in [H,P,S]: a[:,self.numeric_columns_.index(col)] -= self.means_[col]
        dummies = self.encoder_.transform(df[self.cat_columns_]).astype(float)
        base = np.column_stack([a,dummies])
        indexed = {col:base[:,i] for i,col in enumerate(self.base_names_)}
        hc,pc,sc = (indexed[f'{col}_centered'] for col in [H,P,S])
        terms=[base]
        if self.model>=1: terms += [np.column_stack([hc*indexed[k] for k in self.k_names_])]
        if self.model>=2: terms += [np.column_stack([hc*indexed[u] for u in self.u_names_])]
        if self.model>=3: terms += [np.column_stack([indexed[u]*indexed[k] for u in self.u_names_ for k in self.k_names_])]
        if self.model>=4: terms += [np.column_stack([hc*pc,hc*sc])]
        result = np.column_stack(terms)
        assert result.shape[1] == len(self.feature_names_)
        return result

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_,dtype=object)

def make_model(version=3, drop_columns=()):
    # Không StandardScaler trong benchmark: hệ số và intercept trong đơn vị gốc,
    # các continuous H/P/S được CENTER mà KHÔNG chia standard deviation.
    return Pipeline([('features',InteractionFeatures(version,drop_columns)),
                     ('ols',LinearRegression(fit_intercept=True))])

print('Đã định nghĩa InteractionFeatures và make_model()')

### 5.1. Kiểm tra tự động và ví dụ tính tay nhỏ

Trước khi train 50.000 mẫu, cần kiểm chứng bằng một ví dụ giả định rất nhỏ:

- `Hc = H − mean_train(H)`, không thể lấy mean validation/test.
- Với dummy \(K=1\), interaction phải đúng \(H_c K\); tương tự \(H_cU\), \(UK\).
- I0–I4 giữ tất cả main effects, không thêm cả raw H và Hc trùng nhau.

Ví dụ toy test dưới chỉ kiểm thử công thức, **không phải metric Lab 01**.

In [ ]:
# Toy data lấy vài hàng TRAIN đã có labels và mức categories, nhưng không dùng val/test.
toy = train[INPUTS].head(20).copy()
# Bảo đảm có ít nhất 2 mức cho category, bằng lấy thêm ngẫu nhiên từ train nếu cần.
toy = pd.concat([toy,train[INPUTS].sample(n=min(600,len(train)),random_state=SEED)],ignore_index=True)
trans = InteractionFeatures(model=3).fit(toy)
row = toy.iloc[[0]].copy()
z = trans.transform(row)[0]
idx = {f:i for i,f in enumerate(trans.get_feature_names_out())}
hc = float(row.iloc[0][H]) - trans.means_[H]
assert np.isclose(z[idx[H+'_centered']],hc)
for k in trans.k_names_:
    assert np.isclose(z[idx['Hc*'+k]], hc*z[idx[k]])
for u in trans.u_names_:
    assert np.isclose(z[idx['Hc*'+u]], hc*z[idx[u]])
    for k in trans.k_names_:
        assert np.isclose(z[idx[u+'*'+k]],z[idx[u]]*z[idx[k]])
print('PASS: toy check Hc, Hc*K, Hc*U, U*K')

term_counts=[]
for version, expected in EXPECTED_TERMS.items():
    f=InteractionFeatures(model=version).fit(train[INPUTS])
    actual=len(f.get_feature_names_out())
    term_counts.append({'Model':f'I{version}','terms':actual,'expected':expected})
    if actual != expected:
        raise ValueError(f'I{version}: {actual} terms, expected {expected}; kiểm tra levels/encoding trong TRAIN.')
display(pd.DataFrame(term_counts))
print('Reference levels I3:', InteractionFeatures(model=3).fit(train[INPUTS]).references_)

## 6. Training protocol: chỉ train và validation, chưa đụng test

- **5-fold KFold CV chỉ trên train**, cùng `seed=42` và cùng fold assignment cho các cấu hình.
- Mỗi lần fit model: Transformer và OLS **fit trên 4 folds**, dự đoán fold thứ năm; tránh data leakage.
- Lặp lại với I0–I4. Tổng hợp RMSE/MAE/MSE/R² của 5 folds bằng mean ± sample SD.
- Fit một bản trên toàn bộ train để đánh giá **validation riêng**, không fit từ validation.
- Model selection dựa **CV mean RMSE**; trường hợp sai số cách nhau trong 0.1%, ưu tiên ít terms hơn. Test tuyệt đối không tham gia lựa chọn.
- **Ablation I3**: đầy đủ, bỏ `Anxiety_Level_During_Exams`, bỏ cả nhóm D. Các phiên bản ablation dùng chính xác cùng các folds.


### Các chỉ số đánh giá mô hình

**Mean Absolute Error (MAE):**

$$
\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|
$$

**Mean Squared Error (MSE):**

$$
\mathrm{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

**Root Mean Squared Error (RMSE):**

$$
\mathrm{RMSE} = \sqrt{\mathrm{MSE}}
$$

**Coefficient of Determination (R²):**

$$
R^2 = 1-\frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}{\sum_{i=1}^{n}(y_i-\bar{y})^2}
$$


**RMSE/MAE thấp hơn là tốt hơn; R² cao hơn là tốt hơn. R²=0.90 không phải "accuracy=90%".**

In [ ]:
def metrics(y_true,y_pred):
    y_true,y_pred = np.asarray(y_true,float), np.asarray(y_pred,float)
    mse=float(mean_squared_error(y_true,y_pred))
    return {'mae':float(mean_absolute_error(y_true,y_pred)),
            'mse':mse, 'rmse':float(np.sqrt(mse)),
            'r2':float(r2_score(y_true,y_pred))}

def development(train_df,val_df):
    ytr=train_df[TARGET].astype(float).to_numpy()
    yval=val_df[TARGET].astype(float).to_numpy()
    configs={f'I{i}':(i,()) for i in range(5)}
    configs.update({'I3_no_Anxiety':(3,('Anxiety_Level_During_Exams',)),
                    'I3_no_D':(3,('Perceived_AI_Dependency','Anxiety_Level_During_Exams'))})
    folds=list(KFold(n_splits=5,shuffle=True,random_state=SEED).split(train_df))
    assignment=np.empty(len(train_df),dtype=int)
    for fid,(_,v_idx) in enumerate(folds):assignment[v_idx]=fid
    fold_assignment=pd.DataFrame({'Student_ID':train_df['Student_ID'],'fold':assignment})
    fold_assignment.to_csv(TABLES/'train_cv_fold_assignments.csv',index=False)
    cv_rows,comparison_rows,models,val_preds={},{},{},{}
    fold_metrics=[]
    for name,(version,drop) in configs.items():
        rows=[]
        for fid,(tr_idx,cv_idx) in enumerate(folds):
            fold_model=make_model(version,drop)
            fold_model.fit(train_df.iloc[tr_idx][INPUTS],ytr[tr_idx])
            pred=fold_model.predict(train_df.iloc[cv_idx][INPUTS])
            met=metrics(ytr[cv_idx],pred)
            fold_metrics.append({'model':name,'fold':fid+1,'n':len(cv_idx),**met})
            rows.append(met)
        fitted=make_model(version,drop).fit(train_df[INPUTS],ytr)
        models[name]=fitted
        val_preds[name]=fitted.predict(val_df[INPUTS])
        vm=metrics(yval,val_preds[name])
        agg={'model':name,'terms':len(fitted.named_steps['features'].get_feature_names_out()),
             **{f'cv_{k}_mean':float(np.mean([r[k] for r in rows])) for k in ['mae','mse','rmse','r2']},
             **{f'cv_{k}_sd':float(np.std([r[k] for r in rows],ddof=1)) for k in ['mae','mse','rmse','r2']},
             **{f'val_{k}':v for k,v in vm.items()}}
        comparison_rows[name]=agg
        print(f'{name:<15} | CV RMSE {agg["cv_rmse_mean"]:.5f} ± {agg["cv_rmse_sd"]:.5f} | Val RMSE {agg["val_rmse"]:.5f}')
    fold_df=pd.DataFrame(fold_metrics)
    comparison=pd.DataFrame(comparison_rows.values())
    choices=comparison[comparison.model.isin([f'I{i}' for i in range(5)])]
    best=float(choices.cv_rmse_mean.min())
    near=choices[choices.cv_rmse_mean<=best*1.001]
    selected=near.sort_values(['terms','cv_rmse_mean']).iloc[0]['model']
    return fold_df,comparison,models,val_preds,selected

folds,comparison,models,valpred,selected = development(train,val)
folds.to_csv(TABLES/'cv_folds.csv',index=False)
comparison.to_csv(TABLES/'development_comparison.csv',index=False)
display(comparison.round(6))
print('Chọn theo 5-fold CV (KHÔNG theo test):',selected)

## 7. So sánh hiệu năng và ablation

Biểu đồ RMSE có thanh sai số thể hiện **độ lệch chuẩn giữa 5 folds**, không phải confidence interval. Chênh lệch nhỏ so với SD cần diễn giải thận trọng; chỉ từ biểu đồ chưa thể kết luận khác biệt có ý nghĩa thống kê.

In [ ]:
main = comparison[comparison.model.isin([f'I{i}' for i in range(5)])].sort_values('model')
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(main.model,main.cv_rmse_mean,yerr=main.cv_rmse_sd,capsize=4)
ax.set(title='05 · CV RMSE (mean ± SD), chỉ TRAIN',xlabel='Model',ylabel='CV RMSE')
show(fig,'05_cv_rmse',f"CV tốt nhất: {main.loc[main.cv_rmse_mean.idxmin(),'model']}, RMSE={main.cv_rmse_mean.min():.5f}.")

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
ax.bar(main.model,main.val_rmse)
ax.set(title='06 · Validation RMSE — chưa dùng TEST',xlabel='Model',ylabel='Validation RMSE')
show(fig,'06_val_rmse',f"Validation RMSE thấp nhất: {main.val_rmse.min():.5f}. Không chọn lại model chỉ vì cột này.")

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(main.model,main.cv_rmse_mean,marker='o')
ax.set(title='07 · Ablation lồng nhau I0–I4 (train CV)',xlabel='Feature set',ylabel='CV RMSE')
show(fig,'07_incremental_interactions',f"I0→I3 chênh CV RMSE {main.set_index('model').loc['I0','cv_rmse_mean']-main.set_index('model').loc['I3','cv_rmse_mean']:+.5f}.")

print('Group D / no-Anxiety sensitivity (cùng split/folds):')
display(comparison[comparison.model.str.contains('I3')][['model','terms','cv_rmse_mean','cv_rmse_sd','val_rmse','val_mae','val_r2']].round(6))

### 7.1. Lý do chọn I3 thay vì I4

**Mục tiêu lựa chọn:** cân bằng giữa khả năng dự đoán ngoài mẫu, độ phức tạp và khả năng giải thích của mô hình (nguyên tắc *parsimony*). Mô hình có nhiều tương tác hơn không nhất thiết là lựa chọn tốt hơn nếu mức cải thiện dự đoán rất nhỏ.

#### a. Khác biệt về cấu trúc

I3 có 53 predictor terms, bao gồm ba nhóm tương tác chính: AI Hours × Prompt Skill, AI Hours × Use Case và Use Case × Prompt Skill. I4 có 55 terms, bổ sung hai tương tác \(H_cP_c\) (AI Hours × Pre GPA) và \(H_cS_c\) (AI Hours × Traditional Study Hours). Cả hai đều dùng OLS; các hệ số được ước lượng riêng cho từng cấu hình.

I4 có phạm vi biểu diễn rộng hơn: độ dốc theo AI Hours có thể thay đổi thêm theo GPA đầu kỳ và số giờ học truyền thống. I3 vẫn giữ main effects của hai biến này, chỉ không mô hình hóa hai tương tác bổ sung. Vì vậy, chọn I3 không có nghĩa là loại bỏ Pre GPA hoặc Study Hours khỏi đầu vào.

#### b. Bằng chứng thực nghiệm

Bảng dưới minh họa kết quả của lần chạy trước đã được ghi nhận khi thảo luận notebook (không phải output đã lưu trong phiên bản file hiện tại). Sau khi chạy lại, đối chiếu với bảng `comparison` ở phần 6 để xác nhận các giá trị và model được chọn.

| Chỉ số | I3 | I4 |
|---|---:|---:|
| Predictor terms | 53 | 55 |
| CV RMSE mean | 0.150496 | 0.150477 |
| CV RMSE sample SD | 0.001965 | 0.001960 |
| Validation RMSE | 0.150687 | 0.150608 |
| Validation R² | 0.908234 | 0.908330 |

I4 có CV RMSE thấp hơn I3 khoảng \(0.000019\) GPA, tương ứng mức giảm tương đối so với I3:

\[
\frac{0.150496-0.150477}{0.150496}\times100
\approx 0.0126\%.
\]

Validation RMSE cũng giảm, nhưng chỉ khoảng \(0.000079\) GPA. Kết quả cho thấy I4 có cải thiện về mặt số học, song mức cải thiện quan sát được rất nhỏ so với việc bổ sung hai terms. Chênh lệch nhỏ hơn độ lệch chuẩn RMSE giữa các folds; điều này gợi ý cần diễn giải thận trọng, **không phải** bằng chứng rằng hai mô hình tương đương hoặc khác biệt không có ý nghĩa thống kê. Muốn kiểm định cần phân tích chênh lệch theo từng fold với phương pháp phù hợp với sự phụ thuộc giữa các folds.

#### c. Áp dụng quy tắc lựa chọn ở phần 6

Notebook tìm CV RMSE thấp nhất trong I0–I4, sau đó giữ các cấu hình thỏa:

\[
\mathrm{CV\ RMSE}_{m}\leq 1.001\times \min_{k\in\{I0,I1,I2,I3,I4\}}\mathrm{CV\ RMSE}_{k}.
\]

Trong nhóm này, ưu tiên mô hình có ít predictor terms nhất; nếu số terms bằng nhau, ưu tiên CV RMSE thấp hơn.

Với số liệu minh họa trên, I4 là mô hình có CV RMSE thấp nhất, nên ngưỡng chấp nhận là:

\[
0.150477\times 1.001=0.150627477.
\]

I3 có CV RMSE \(0.150496\), nằm trong ngưỡng này và chỉ có 53 terms thay vì 55. Theo quy tắc đã đặt ra, kết quả lựa chọn là I3 (các cấu hình I0–I2 không nằm trong nhóm gần tối ưu ở lần chạy trước). Ngưỡng 0,1% là tiêu chí thực nghiệm của notebook, không phải định luật thống kê; validation chỉ dùng để đối chiếu, và test không tham gia lựa chọn.

#### d. Khả năng giải thích và giới hạn của quyết định

Với I3, độ dốc GPA dự đoán theo một giờ GenAI, khi giữ các input khác cố định, là:

\[
\frac{\partial\hat y_{I3}}{\partial H}
=\beta_H^{(3)}+\delta_u^{(3)}+\gamma_k^{(3)}.
\]

Với I4, độ dốc có thêm hai thành phần:

\[
\frac{\partial\hat y_{I4}}{\partial H}
=\beta_H^{(4)}+\delta_u^{(4)}+\gamma_k^{(4)}
+\theta P_c+\phi S_c.
\]

I3 thuận tiện hơn khi trình bày conditional slopes theo Use Case và Prompt Skill vì không cần cố định thêm Pre GPA và Study Hours để xác định độ dốc. Điều này phù hợp với các phân tích chuyên sâu ở phần 8–11. I4 vẫn hữu ích như cấu hình mở rộng để nghiên cứu hai tương tác liên tục–liên tục; việc chọn I3 không chứng minh hai tương tác đó vô ích, không chứng minh I4 bị overfitting và không chứng minh I4 có đa cộng tuyến cao hơn (phần 11 chỉ chẩn đoán I3).

**Kết luận:** Với kết quả đã quan sát và tiêu chí lựa chọn hiện tại, I3 là lựa chọn cân bằng hơn giữa hiệu năng và khả năng giải thích. I4 vẫn được giữ để so sánh và đánh giá đóng góp của hai tương tác bổ sung. I3 là cấu hình nghiên cứu chính của notebook, còn `selected` là cấu hình được chọn tự động từ CV; nếu dữ liệu hoặc kết quả CV thay đổi, `selected` có thể khác I3. Phần 12 vẫn dùng `selected` để final fit và đánh giá trên hold-out test độc lập. Không sử dụng các kết quả này để suy ra quan hệ nhân quả giữa việc dùng GenAI và GPA.

## 8. Giải thích coefficients (không có StandardScaler)

Trong notebook này, OLS dùng continuous features **chỉ center, không chia độ lệch chuẩn**. Vì vậy:

- Hệ số của `Weekly_GenAI_Hours_centered` là **GPA / một giờ AI** cho **nhóm tham chiếu** (khi các interactions còn lại ở 0).
- Hệ số `Hc*Prompt_Engineering_Skill_Beginner` là **mức chênh độ dốc** giữa Beginner và nhóm Prompt Skill tham chiếu.
- Hệ số `Use Case dummy × Prompt Skill dummy` thay đổi mức dự đoán của **tổ hợp**, phải cộng main effects và interaction để lấy contrast đầy đủ.
- Nếu **sau này** thêm `StandardScaler`, hệ số scaled không thể được đọc trực tiếp theo đơn vị giờ: đối với biến liên tục chuẩn hóa \(z=(x-\mu)/\sigma\), hệ số nguyên đơn vị là \(\beta_x=\beta_z/\sigma\). Đối với interaction phải tính theo toàn bộ phép biến đổi, hoặc trực tiếp lấy sai phân dự đoán ở raw input.

In [ ]:
i3 = models['I3']
f3=i3.named_steps['features']
coeff=pd.Series(i3.named_steps['ols'].coef_,index=f3.get_feature_names_out(),name='coefficient')
print('Reference levels:',f3.references_)
print('Intercept:',i3.named_steps['ols'].intercept_)
display(coeff.sort_values(key=lambda x:abs(x),ascending=False).head(18).to_frame())
coeff.to_csv(TABLES/'i3_coefficients.csv',header=True)
fig,ax=plt.subplots(figsize=(9,5))
coeff.reindex(coeff.abs().nlargest(14).index).sort_values().plot.barh(ax=ax)
ax.set(title='08 · I3 coefficients có trị tuyệt đối lớn (đơn vị khác nhau!)',xlabel='Hệ số OLS',ylabel='Feature')
show(fig,'08_i3_coefficients', 'Không so sánh feature importance chỉ bằng |coefficient| khi thang đo khác nhau.')

## 9. Conditional slopes và predicted contrasts

Đối với I3, ở một Use Case \(u\) và Prompt Skill \(k\), nếu giữ các input khác cố định:

\[
\frac{\partial \hat y}{\partial H}=\beta_H+\delta_u+\gamma_k.
\]

Với I4 sẽ cộng thêm \(\theta P_c+\phi S_c\). Khi hai nhóm Use Case/Prompt Skill khác nhau, hiệu dự đoán ở cùng H (***contrast***) còn chịu ảnh hưởng của các main-effect dummy và \(U_iK_j\), không chỉ từ độ dốc.

**Để tránh sai khi diễn giải coefficients,** notebook tính độ dốc bằng \(\hat y(H+1)-\hat y(H)\), trực tiếp qua pipeline. Contrast được tính giữa hai hồ sơ **chỉ khác category cần so sánh** và cùng số giờ AI, GPA đầu kỳ, thời gian học, v.v.

In [ ]:
profile = train.iloc[0][INPUTS].to_dict()
uses = list(map(str,train['Primary_Use_Case'].astype(str).unique()))
skills = list(map(str,train['Prompt_Engineering_Skill'].astype(str).unique()))
uses.sort(); skills.sort()
slopes=pd.DataFrame(index=uses,columns=skills,dtype=float)
base_h=float(f3.means_[H])
for u in uses:
    for k in skills:
        row=profile|{'Primary_Use_Case':u,'Prompt_Engineering_Skill':k,H:base_h}
        row1=row|{H:base_h+1}
        slopes.loc[u,k]=float(i3.predict(pd.DataFrame([row1]))[0]-i3.predict(pd.DataFrame([row]))[0])
slopes.to_csv(TABLES/'i3_conditional_slopes.csv')
display(slopes.round(5))
fig,ax=plt.subplots(figsize=(9,5))
sns.heatmap(slopes.astype(float),annot=True,fmt='.4f',cmap='coolwarm',center=0,ax=ax)
ax.set(title='09 · I3 conditional slopes (GPA / +1 giờ AI)',xlabel='Prompt Skill',ylabel='Use Case')
show(fig,'09_conditional_slopes',f"Slope nhỏ nhất={slopes.min().min():.5f}, lớn nhất={slopes.max().max():.5f}; KHÔNG phải hiệu ứng nhân quả.")
# Kiểm tra slope giải thích hệ số (vì chưa dùng scaling, công thức trực tiếp áp dụng).
coef_H=float(coeff[H+'_centered'])
for u in uses:
    for k in skills:
        expected=coef_H
        if u!=f3.references_['Primary_Use_Case']:
            expected+=float(coeff['Hc*Primary_Use_Case_'+u])
        if k!=f3.references_['Prompt_Engineering_Skill']:
            expected+=float(coeff['Hc*Prompt_Engineering_Skill_'+k])
        assert np.isclose(slopes.loc[u,k],expected,atol=1e-9)
print('PASS: numerical coefficient interpretation == finite-difference predicted slopes')

In [ ]:
# Reference group và contrast tại H = trung bình train, các biến khác giữ nguyên.
ref_u=f3.references_['Primary_Use_Case']
ref_k=f3.references_['Prompt_Engineering_Skill']
ref=profile|{'Primary_Use_Case':ref_u,'Prompt_Engineering_Skill':ref_k,H:base_h}
ref_pred=float(i3.predict(pd.DataFrame([ref]))[0])
contrasts=[]
for u in uses:
    for k in skills:
        row=ref|{'Primary_Use_Case':u,'Prompt_Engineering_Skill':k}
        prediction=float(i3.predict(pd.DataFrame([row]))[0])
        contrasts.append({'use_case':u,'skill':k,'predicted_gpa':prediction,'contrast_vs_reference':prediction-ref_pred})
contrasts=pd.DataFrame(contrasts)
contrasts.to_csv(TABLES/'i3_predicted_contrasts.csv',index=False)
print('Reference:',ref_u,'×',ref_k,'; H trung bình train=',round(base_h,3),'giờ')
display(contrasts.round(5))

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
grid=np.linspace(float(train[H].quantile(.05)),float(train[H].quantile(.95)),40)
for use in uses[:3]:
    for skill in skills:
        profiles=[profile|{H:float(h),'Primary_Use_Case':use,'Prompt_Engineering_Skill':skill} for h in grid]
        predicted=i3.predict(pd.DataFrame(profiles))
        ax.plot(grid,predicted,label=f'{use[:13]} / {skill}',lw=1.5)
ax.set(title='10 · Predicted GPA curves (I3; profile còn lại cố định)',xlabel='Giờ GenAI/tuần',ylabel='Predicted Post GPA')
ax.legend(loc='best',fontsize=8,ncol=2)
show(fig,'10_predicted_curves',f'Vẽ trong vùng 5–95 percentile của giờ AI: {grid.min():.2f}–{grid.max():.2f} giờ.')

## 10. Sai số theo subgroup và residual (trên validation)

Kiểm tra RMSE theo `Primary_Use_Case`, theo `Prompt_Engineering_Skill` và theo từng khoảng AI Hours. **Luôn ghi n**: RMSE dao động mạnh ở nhóm ít mẫu. Biểu đồ residual giúp phát hiện mô hình dự đoán lệch ở một vùng GPA hoặc có hiện tượng phương sai sai số thay đổi.

In [ ]:
def subgroup_report(frame, pred, group_col):
    local=frame[[group_col,TARGET]].copy()
    local['prediction']=np.asarray(pred,float)
    records=[]
    for key,grp in local.groupby(group_col):
        met=metrics(grp[TARGET],grp['prediction'])
        records.append({'group':str(key),'n':len(grp),**met})
    return pd.DataFrame(records).sort_values('rmse',ascending=False)

sub_use=subgroup_report(val,valpred['I3'],'Primary_Use_Case')
sub_use.to_csv(TABLES/'i3_val_subgroup_use_case.csv',index=False)
display(sub_use.round(4))
fig,ax=plt.subplots(figsize=(9,5))
ax.barh(sub_use['group'][::-1],sub_use['rmse'][::-1])
ax.set(title='11 · I3 validation RMSE theo Use Case',xlabel='RMSE',ylabel='Use Case')
show(fig,'11_subgroup_use_case',f"Khoảng RMSE {sub_use.rmse.min():.4f}–{sub_use.rmse.max():.4f}; n nhỏ nhất={sub_use.n.min()}.")

In [ ]:
sub_skill=subgroup_report(val,valpred['I3'],'Prompt_Engineering_Skill')
sub_skill.to_csv(TABLES/'i3_val_subgroup_prompt_skill.csv',index=False)
display(sub_skill.round(4))
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(sub_skill.group,sub_skill.rmse)
ax.set(title='12 · I3 validation RMSE theo Prompt Skill',xlabel='Prompt Skill',ylabel='RMSE')
show(fig,'12_subgroup_prompt_skill',f"Khoảng RMSE {sub_skill.rmse.min():.4f}–{sub_skill.rmse.max():.4f}; n nhỏ nhất={sub_skill.n.min()}.")

In [ ]:
# 4 nhóm (theo quantile từ TRAIN; validation chỉ transform bằng cạnh train).
edge=np.unique(train[H].quantile([0,.25,.5,.75,1]).to_numpy())
if len(edge)>=3:
    val_range=val.copy()
    val_range['ai_hour_segment']=pd.cut(val_range[H],bins=np.r_[-np.inf,edge[1:-1],np.inf],include_lowest=True)
    sub_hour=subgroup_report(val_range,valpred['I3'],'ai_hour_segment')
    sub_hour.to_csv(TABLES/'i3_val_subgroup_hours.csv',index=False)
    display(sub_hour.round(4))
    fig,ax=plt.subplots(figsize=(8,4))
    ax.bar(sub_hour.group,sub_hour.rmse)
    ax.tick_params(axis='x',rotation=25)
    ax.set(title='13 · I3 validation RMSE theo khoảng AI Hours',xlabel='Khoảng giờ AI (train quantiles)',ylabel='RMSE')
    show(fig,'13_subgroup_ai_hours',f"Số nhóm {len(sub_hour)}, mỗi nhóm n >= {sub_hour.n.min()}.")
else: print('Không đủ khoảng AI Hours trên train để phân nhóm.')

In [ ]:
yp_val=np.asarray(valpred['I3'],float)
resid_val=val[TARGET].to_numpy(float)-yp_val
fig,ax=plt.subplots(figsize=(7,4))
ax.scatter(yp_val,resid_val,alpha=.17,s=8)
ax.axhline(0,color='red',linewidth=1)
ax.set(title='14 · I3 validation: residual vs predicted',xlabel='Predicted Post GPA',ylabel='Actual − predicted')
show(fig,'14_validation_residuals',f"Residual mean={resid_val.mean():.5f}; std={resid_val.std():.5f}.")

In [ ]:
fig,ax=plt.subplots(figsize=(6,5))
ax.scatter(val[TARGET],yp_val,s=8,alpha=.14)
lims=[float(min(val[TARGET].min(),yp_val.min())),float(max(val[TARGET].max(),yp_val.max()))]
ax.plot(lims,lims,'r--',linewidth=1,label='y=x')
ax.set(title='15 · I3 validation: actual vs predicted',xlabel='Actual Post GPA',ylabel='Predicted Post GPA')
ax.legend()
show(fig,'15_validation_actual_pred',f"MAE={metrics(val[TARGET],yp_val)['mae']:.4f}; RMSE={metrics(val[TARGET],yp_val)['rmse']:.4f}.")

## 11. Chẩn đoán đa cộng tuyến: condition number và VIF

VIF của cột \(x_j\) có thể viết là \(1/(1-R_j^2)\), với \(R_j^2\) là độ phù hợp khi hồi quy \(x_j\) theo các cột còn lại. **VIF cao cho biết phụ thuộc giữa predictors**, không chứng minh feature gây ra kết quả và không tự động bắt buộc loại biến.

VIF được tính trên **ma trận train đã standardize cho mục đích chẩn đoán**. OLS chính vẫn học trên ma trận chưa chia SD; vì vậy hệ số trong phần 8 không đổi đơn vị.

In [ ]:
X3 = f3.transform(train[INPUTS])
scale=X3.std(axis=0)
active=scale>1e-12
Z=(X3[:,active]-X3[:,active].mean(axis=0))/scale[active]
# SVD trên ma trận 35.000 × ~53, không dùng TEST.
_,sv,vh=np.linalg.svd(Z,full_matrices=False)
sv_tol=np.finfo(float).eps*max(Z.shape)*sv[0]
rank=int((sv>sv_tol).sum())
condition=float(sv.max()/sv.min()) if sv.min()>sv_tol else np.inf
vif_values=(len(Z)*(vh.T**2 / (sv**2)).sum(axis=1)) if rank==Z.shape[1] else np.full(Z.shape[1],np.inf)
vif=pd.DataFrame({'feature':np.asarray(f3.get_feature_names_out())[active], 'vif':vif_values})
vif=vif.sort_values('vif',ascending=False)
vif.to_csv(TABLES/'i3_vif.csv',index=False)
print('Design matrix rank:',rank,'/',Z.shape[1],'| condition number (standardized)=',condition)
display(vif.head(15).round(4))
fig,ax=plt.subplots(figsize=(8,4))
ax.semilogy(sv,marker='.',linestyle='-')
ax.set(title='16 · Singular values của standardized design (I3 train)',xlabel='Thứ tự singular value',ylabel='Singular value (log)')
show(fig,'16_singular_values',f'Rank {rank}/{Z.shape[1]}, condition number ≈ {condition:.3g}.')

In [ ]:
finite=vif[np.isfinite(vif.vif)].head(12)
if len(finite)>0:
    fig,ax=plt.subplots(figsize=(9,5))
    ax.barh(finite.feature[::-1],finite.vif[::-1])
    ax.set(title='17 · Top VIF của Interaction I3 trên train',xlabel='VIF',ylabel='Feature')
    show(fig,'17_vif',f"VIF lớn nhất (trong các giá trị hữu hạn)={finite.vif.max():.3f}.")
else:
    print('VIF plot BLOCKED: standardized design thiếu hạng, toàn bộ VIF gán inf.')

## 12. Final model evaluation — Hold-out test độc lập

- Chọn I0–I4 bằng 5-fold CV trên train, với tie-break 0,1% ưu tiên ít terms. Validation chỉ dùng đánh giá development. Không lựa chọn bằng test.
- Khóa lựa chọn và fit lại toàn bộ pipeline (centering, encoder và OLS) trên **42.500 mẫu train + validation**.
- Mặc định dự đoán **toàn bộ 7.500 mẫu test frozen**, không fit thêm, không clip GPA, không chọn lại mô hình.
- Báo cáo MAE, MSE, RMSE, R², tỷ lệ dự đoán ngoài miền GPA [0,4], sai số theo nhóm và so sánh với baseline dự đoán GPA trung bình của train+validation. Baseline chỉ là mốc tham chiếu, không tham gia chọn mô hình.
- Nếu `LAB01_RUN_FINAL_TEST=0`, chỉ chạy development. Smoke test không được phép tạo kết quả final test.
- Registry ngăn đánh giá lại tự động; sau khi xem test, nếu cần điều chỉnh mô hình, phải có test độc lập mới.

In [ ]:
model_lock={'run_id':RUN_ID,'selected_model':selected,'selection_rule':'5-fold train CV RMSE, tie 0.1% -> fewer terms',
            'selection_date_utc':datetime.now(timezone.utc).isoformat(), 'test_used_to_select':False,
            'frozen_csv_sha256':provenance['sha256'],
            'train_n':len(train),'validation_n':len(val),'test_n':len(frames['test'])}
(OUT/'model_lock.json').write_text(json.dumps(model_lock,indent=2,ensure_ascii=False),encoding='utf-8')
final_fit=pd.concat([train,val],ignore_index=True)
final_model=make_model(int(selected[-1]))
final_model.fit(final_fit[INPUTS],final_fit[TARGET])
joblib.dump(final_model,MODELS/'selected_train_plus_validation.joblib')
print('Selected:',selected,'| final fit n=',len(final_fit),'| model saved to:',MODELS)
print('Final test enabled:', RUN_FINAL_TEST)

In [ ]:
registry_path = OUT_ROOT / 'final_test_registry.json'
test, yp_test = None, None
final_test_metrics = {'status': 'not_executed', 'reason': 'Development mode (LAB01_RUN_FINAL_TEST=0)'}

if RUN_FINAL_TEST:
    if SMOKE_TEST:
        raise RuntimeError('Không cho phép final test trong smoke mode.')
    if registry_path.exists():
        old = json.loads(registry_path.read_text(encoding='utf-8'))
        raise RuntimeError(
            f'Final test đã được ghi nhận (RUN_ID={old.get("run_id")}). '
            f'Xem kết quả tại {old.get("metrics_path")}. '
            'Không chạy lại để tuning; dùng LAB01_RUN_FINAL_TEST=0 cho development.'
        )
    if provenance['counts'] != {'train': 35000, 'validation': 7500, 'test': 7500}:
        raise ValueError('Final test yêu cầu đúng frozen splits 70/15/15.')
    if sha256_file(TEST_CSV) != model_lock['frozen_csv_sha256']['test']:
        raise RuntimeError('Test CSV đã thay đổi sau khi kiểm tra provenance.')

    test = frames['test']
    y_test = test[TARGET].to_numpy(dtype=float)
    yp_test = np.asarray(final_model.predict(test[INPUTS]), dtype=float)
    assert y_test.shape == yp_test.shape == (7500,)
    assert np.isfinite(y_test).all() and np.isfinite(yp_test).all()
    residual = y_test - yp_test

    test_summary = metrics(y_test, yp_test)
    # Đối chiếu cách tính metric bằng NumPy (không dùng model khác để chọn lại).
    np.testing.assert_allclose(test_summary['mae'], np.mean(np.abs(residual)))
    np.testing.assert_allclose(test_summary['mse'], np.mean(residual**2))
    np.testing.assert_allclose(test_summary['rmse']**2, test_summary['mse'])

    baseline_mean = float(final_fit[TARGET].mean())
    baseline_summary = metrics(y_test, np.full(len(test), baseline_mean))
    final_test_metrics = {
        'status': 'executed', 'n': len(test), **test_summary,
        'pred_below_0_pct': float(100*np.mean(yp_test < 0)),
        'pred_above_4_pct': float(100*np.mean(yp_test > 4)),
        'mean_residual': float(residual.mean()),
        'baseline_train_val_mean': baseline_mean,
        'baseline_metrics': baseline_summary
    }
    predictions = pd.DataFrame({
        'Student_ID': test['Student_ID'], 'actual': y_test,
        'predicted': yp_test, 'residual': residual,
        'absolute_error': np.abs(residual)
    })
    predictions.to_csv(TABLES / 'final_test_predictions.csv', index=False)
    test_comparison = pd.DataFrame([
        {'model': selected, 'n': len(test), **test_summary},
        {'model': 'Mean baseline (train+validation)', 'n': len(test), **baseline_summary}
    ])
    test_comparison.to_csv(TABLES / 'final_test_comparison.csv', index=False)
    print('FINAL TEST METRICS (hold-out):')
    display(test_comparison.round(6))

    for group_col in ['Primary_Use_Case', 'Prompt_Engineering_Skill']:
        report = subgroup_report(test, yp_test, group_col)
        report.to_csv(TABLES / f'final_test_subgroup_{group_col}.csv', index=False)
        display(report.round(6))
else:
    print('Final test chưa chạy: development mode hoặc smoke mode.')

all_metrics = {
    'run_id': RUN_ID, 'selected_model': selected,
    'train_cv_selection': comparison.loc[comparison.model.eq(selected)].to_dict('records')[0],
    'validation': metrics(val[TARGET], valpred[selected]),
    'final_test': final_test_metrics, 'smoke_test': SMOKE_TEST
}
metrics_path = OUT / 'metrics.json'
metrics_path.write_text(json.dumps(all_metrics, indent=2, ensure_ascii=False), encoding='utf-8')

if final_test_metrics['status'] == 'executed':
    # Ghi registry sau khi dự đoán và metrics đã được lưu thành công.
    with registry_path.open('x', encoding='utf-8') as stream:
        json.dump({
            'run_id': RUN_ID, 'selected': selected,
            'frozen_test_sha256': provenance['sha256']['test'],
            'metrics_path': str(metrics_path),
            'recorded_utc': datetime.now(timezone.utc).isoformat()
        }, stream, indent=2, ensure_ascii=False)

In [ ]:
if test is not None and final_test_metrics.get('status')=='executed':
    residual=test[TARGET].to_numpy()-yp_test
    fig,ax=plt.subplots(figsize=(7,4))
    ax.scatter(test[TARGET],yp_test,s=8,alpha=.16)
    lims=[min(float(test[TARGET].min()),float(yp_test.min())),max(float(test[TARGET].max()),float(yp_test.max()))]
    ax.plot(lims,lims,'r--',lw=1)
    ax.set(title='18 · Final test: actual vs predicted',xlabel='Actual Post GPA',ylabel='Predicted GPA')
    show(fig,'18_final_test_actual_pred',f'Final test RMSE={final_test_metrics["rmse"]:.5f}; chỉ báo cáo, KHÔNG dùng lựa chọn lại mô hình.')
    fig,ax=plt.subplots(figsize=(7,4))
    ax.scatter(yp_test,residual,s=8,alpha=.16)
    ax.axhline(0,color='red',linewidth=1)
    ax.set(title='19 · Final test: residual vs fitted',xlabel='Predicted GPA',ylabel='Actual − predicted')
    show(fig,'19_final_test_residual',f'Mean residual={residual.mean():.5f}.')
else:
    print('Không tạo đồ thị test. Các biểu đồ validation ở trên vẫn có thể dùng báo cáo development.')

## 13. Kiểm thử chức năng inference và demo dự đoán GPA

Đây là kiểm thử chức năng ứng dụng, không thay thế hold-out test ở phần 12.

**Input:** dictionary đúng 12 raw features, category hợp lệ. **Output:** GPA dự đoán dạng `float` và cảnh báo nếu ngoài [0,4]. Không tự nhập one-hot/interaction và không clip.

Kiểm tra đầu ra hữu hạn, nhất quán với pipeline đã fit, tính lặp lại và từ chối input thiếu/thừa, NaN hoặc category chưa gặp. Profile minh họa lấy từ train, không được dùng để báo cáo hiệu năng test.

In [ ]:
def predict_student(student, model=None):
    if model is None: model = final_model
    if not isinstance(student,dict): raise TypeError('student phải là dict raw features')
    missing=set(INPUTS)-set(student)
    extras=set(student)-set(INPUTS)
    if missing or extras:
        raise ValueError(f'Input phải có 12 raw features; thiếu {sorted(missing)}, dư {sorted(extras)}')
    prediction=float(model.predict(pd.DataFrame([student]))[0])
    if not np.isfinite(prediction): raise ValueError('Dự đoán không hữu hạn')
    result={'predicted_post_semester_gpa':prediction,'out_of_range':not (0<=prediction<=4)}
    if result['out_of_range']:
        result['warning']='GPA ngoài miền [0,4]: OLS không có ràng buộc; không clip kết quả.'
    return result

# Default hợp lệ lấy từ một hàng của TRAIN, nhưng đây chỉ là profile minh họa.
example=train.iloc[0][INPUTS].to_dict()
result=predict_student(example)
print('Ví dụ student raw features:',example)
print('Dự đoán:',result)

# Functional tests: không fit lại và không dùng nhãn test.
reference = float(final_model.predict(pd.DataFrame([example], columns=INPUTS))[0])
assert isinstance(result['predicted_post_semester_gpa'], float)
np.testing.assert_allclose(result['predicted_post_semester_gpa'], reference)
assert result == predict_student(example.copy())
assert result['out_of_range'] == (not 0 <= reference <= 4)

def expect_error(profile, error_type):
    try:
        predict_student(profile)
    except error_type:
        return
    raise AssertionError(f'Expected {error_type.__name__} for invalid input')

expect_error([], TypeError)
expect_error({k: v for k, v in example.items() if k != H}, ValueError)
expect_error(example | {TARGET: 3.0}, ValueError)
expect_error(example | {H: float('nan')}, ValueError)
expect_error(example | {'Primary_Use_Case': '__unseen_category__'}, ValueError)
inference_checks_passed = True
print('PASS: inference contract, repeatability, invalid inputs')

## 14. What-if analysis và kiểm thử tính nhất quán

Chỉ thay `Weekly_GenAI_Hours` của cùng profile, giữ 11 trường còn lại cố định. Với I0–I4, khi giữ các biến khác cố định, dự đoán là hàm affine theo H; vì vậy có thể kiểm tra độ dốc không đổi trên lưới giờ và đối chiếu batch inference với API dự đoán từng mẫu.

Đây là predicted association có điều kiện, không phải ước lượng hiệu ứng nhân quả hoặc test accuracy.

In [ ]:
whatif_hours=np.linspace(float(train[H].quantile(.05)),float(train[H].quantile(.95)),40)
whatif_profiles=pd.DataFrame([example|{H:float(h)} for h in whatif_hours])
whatif_preds=final_model.predict(whatif_profiles)
# Kiểm tra tính nhất quán của what-if, không sử dụng test labels.
assert len(whatif_preds) == len(whatif_hours) and np.isfinite(whatif_preds).all()
for col in INPUTS:
    if col != H:
        assert whatif_profiles[col].eq(example[col]).all()
np.testing.assert_allclose(
    whatif_preds,
    [predict_student(row)['predicted_post_semester_gpa'] for row in whatif_profiles.to_dict('records')],
    rtol=1e-10, atol=1e-10
)
if np.ptp(whatif_hours) > 0:
    slopes_check = np.diff(whatif_preds) / np.diff(whatif_hours)
    np.testing.assert_allclose(slopes_check, slopes_check[0], rtol=1e-7, atol=1e-9)
whatif_checks_passed = True
pd.DataFrame({H: whatif_hours, 'predicted_gpa': whatif_preds}).to_csv(
    TABLES / 'what_if_predictions.csv', index=False)
print('PASS: what-if consistency')
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(whatif_hours,whatif_preds,lw=2)
ax.axhline(4,color='red',linestyle=':',linewidth=1,label='GPA 4.0')
ax.set(title=f'20 · What-if: tăng/giảm GenAI Hours ({selected})',xlabel='Giờ GenAI/tuần',ylabel='Predicted Post GPA')
ax.legend()
show(fig,'20_what_if',f'Dự đoán nằm trong khoảng {whatif_preds.min():.3f}–{whatif_preds.max():.3f} trên lưới giờ minh họa.')

### 14.1. Widget form (tùy chọn) và fallback không cần widget

Nếu môi trường có `ipywidgets`, form cho phép chỉnh **toàn bộ 12 raw inputs** rồi bấm nút để dự đoán. Nếu chưa cài widget hoặc notebook được chạy headless (nbconvert), `predict_student(example)` phía trên **vẫn chạy bình thường**. Không được để dependency giao diện làm thất bại bài huấn luyện.

In [ ]:
try:
    import ipywidgets as widgets
    controls={}
    for col in INPUTS:
        if col in CATEGORICAL:
            options=sorted(train[col].unique().tolist())
            controls[col]=widgets.Dropdown(options=options,value=example[col],description=col[:19],style={'description_width':'initial'})
        elif col=='Paid_Subscription':
            controls[col]=widgets.Checkbox(value=bool(example[col]),description=col)
        else:
            controls[col]=widgets.FloatText(value=float(example[col]),description=col[:19],style={'description_width':'initial'})
    btn=widgets.Button(description='Dự đoán GPA',button_style='info')
    ui_out=widgets.Output()
    def handle_click(_):
        new_profile={k:w.value for k,w in controls.items()}
        with ui_out:
            ui_out.clear_output(wait=True)
            print(predict_student(new_profile))
    btn.on_click(handle_click)
    display(widgets.VBox(list(controls.values())+[btn,ui_out]))
    print('Widget đã tạo. Dùng nút Dự đoán GPA hoặc gọi predict_student(dict_12_features).')
except ImportError:
    print('ipywidgets không được cài; sử dụng hàm predict_student(example) ở trên là đủ.')

## 15. Kiểm tra và bàn giao kết quả

Mỗi lần chạy có RUN_ID riêng. Kiểm tra các artifact được tạo trong run hiện tại: model, metrics, provenance, bảng CV/validation, biểu đồ và kết quả functional tests. Với final test, bắt buộc có bảng dự đoán đúng 7.500 ID, bảng metric, các báo cáo subgroup, hai biểu đồ test và registry khớp RUN_ID.

Đối chiếu metrics từ file dự đoán đã lưu (không predict lại trên test). Chỉ dùng kết quả final test hợp lệ cho báo cáo; development/smoke không thay thế final test.

In [ ]:
assert inference_checks_passed and whatif_checks_passed
assert len(FIGURES_CREATED) == len(set(FIGURES_CREATED))
assert len(FIGURES_CREATED) >= 10
assert all(Path(x).is_file() and Path(x).parent == FIG for x in FIGURES_CREATED)
required_artifacts = [
    OUT / 'split_provenance.json', OUT / 'model_lock.json', OUT / 'metrics.json',
    TABLES / 'development_comparison.csv', TABLES / 'cv_folds.csv',
    TABLES / 'what_if_predictions.csv', MODELS / 'selected_train_plus_validation.joblib'
]
if RUN_FINAL_TEST:
    assert final_test_metrics['status'] == 'executed' and not SMOKE_TEST
    required_artifacts += [
        TABLES / 'final_test_predictions.csv', TABLES / 'final_test_comparison.csv',
        TABLES / 'final_test_subgroup_Primary_Use_Case.csv',
        TABLES / 'final_test_subgroup_Prompt_Engineering_Skill.csv',
        FIG / '18_final_test_actual_pred.png', FIG / '19_final_test_residual.png',
        registry_path
    ]
    saved = pd.read_csv(TABLES / 'final_test_predictions.csv', dtype={'Student_ID': str})
    assert len(saved) == 7500 and saved['Student_ID'].is_unique
    assert saved['Student_ID'].tolist() == test['Student_ID'].astype(str).tolist()
    np.testing.assert_allclose(saved['actual'], test[TARGET].to_numpy(dtype=float))
    np.testing.assert_allclose(saved['residual'], saved['actual'] - saved['predicted'], atol=1e-10)
    stored_metrics = json.loads((OUT / 'metrics.json').read_text(encoding='utf-8'))['final_test']
    for key, value in metrics(saved['actual'], saved['predicted']).items():
        np.testing.assert_allclose(stored_metrics[key], value, atol=1e-10)
    assert json.loads(registry_path.read_text(encoding='utf-8'))['run_id'] == RUN_ID
assert all(path.is_file() and path.stat().st_size > 0 for path in required_artifacts)
manifest = {
    'run_id': RUN_ID, 'started_with_frozen_csv': True, 'smoke_test': SMOKE_TEST,
    'test_status': final_test_metrics['status'], 'csv_sha256': provenance['sha256'],
    'selected': selected, 'created_figures': FIGURES_CREATED,
    'figure_count': len(FIGURES_CREATED),
    'functional_tests': {'inference': 'passed', 'what_if': 'passed', 'artifacts': 'passed'},
    'required_artifacts': [str(p) for p in required_artifacts],
    'created_utc': datetime.now(timezone.utc).isoformat()
}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print('RUN_ID:', RUN_ID)
print('PASS: artifact verification | figures:', len(FIGURES_CREATED))
print('Metrics:', OUT / 'metrics.json')
print('Bảng:', TABLES)
print('Model:', MODELS / 'selected_train_plus_validation.joblib')
print('Final test:', final_test_metrics['status'])
if SMOKE_TEST:
    print('Smoke mode: không dùng metrics cho báo cáo.')
elif not RUN_FINAL_TEST:
    print('Development mode: chưa đánh giá final test.')
else:
    print('Hoàn thành final evaluation trên 7.500 mẫu frozen.')

## 16. Những câu hỏi tự kiểm tra trước khi bảo vệ

1. Vì sao thêm \(H_c K\) nhưng vẫn gọi là Linear Regression? *Vì hàm dự đoán tuyến tính theo hệ số, không nhất thiết tuyến tính theo feature thô.*
2. Vì sao dùng 2 dummy cho Prompt Skill 3 mức? *Vì một mức làm reference; tránh dummy trap khi có intercept.*
3. I3 khác I2 ở điểm nào? *Thêm 8 terms Use Case × Prompt Skill, có thể thay đổi dự đoán theo tổ hợp nhóm.*
4. Độ dốc theo AI Hours của một nhóm là gì? *\(\beta_H+\delta_u+\gamma_k\) cho I3; có thể kiểm bằng \(\hat y(H+1)-\hat y(H)\).* 
5. Tại sao không dùng test khi chọn model? *Vì như vậy test không còn là đánh giá ngoài mẫu độc lập.*
6. Vì sao hệ số của feature lớn không luôn là feature quan trọng? *Đơn vị feature khác nhau, interaction làm hệ số phụ thuộc nhóm tham chiếu.*
7. Vì sao RMSE của nhóm nhỏ có thể dao động mạnh? *Ít mẫu, nhạy với outliers và không đủ đại diện.*
8. Nếu GPA dự đoán > 4 thì sao? *Cảnh báo, báo cáo tỷ lệ; không tự ý sửa kết quả mà không ghi lại thí nghiệm.*
9. Nếu bỏ Anxiety giúp giảm accuracy? *Nêu trade-off: đổi độ sẵn có của dữ liệu lấy một phần hiệu năng; không giả định feature có ở prediction cutoff.*
10. **Không kết luận** tăng giờ dùng AI chắc chắn cải thiện GPA; đây là dự đoán có điều kiện trên dữ liệu synthetic.

**Tham khảo:** scikit-learn User Guide — Linear Models, Pipeline, Cross-Validation, OneHotEncoder, metrics. Nội dung và hình trong notebook được sinh từ ba CSV người dùng cung cấp tại máy chạy.